In [ ]:
import pandas as pd
import geopandas as gpd
import mapclassify
import matplotlib.pyplot as plt
import textwrap
import matplotlib.patches as mpatches

import sys
import os

sys.path.append(os.path.abspath(".."))
from utils.plot_style import apply_plot_style
from utils.config import root_dir

apply_plot_style()


# function to create a 4-grid legend
def insert_bivariate_legend(legend_ax, color_mapping):
    # Corrected grid positions
    positions = {
        "0-0": (0, 0),  # low-low
        "0-1": (1, 0),  # low-high
        "1-0": (0, 1),  # high-low
        "1-1": (1, 1),  # high-high
    }

    # Add rectangles for each grid cell
    for key, (row, col) in positions.items():
        rect = mpatches.Rectangle(
            (col, row),
            1,
            1,
            facecolor=color_mapping[key],
            edgecolor="white",
            linewidth=2,
        )
        legend_ax.add_patch(rect)

    # Set axis limits
    legend_ax.set_xlim(0, 2)
    legend_ax.set_ylim(0, 2)
    legend_ax.set_aspect("equal", adjustable="box")

    # Add axis break point labels
    legend_ax.text(0.5, -0.1, "= 0", ha="center", va="center")
    legend_ax.text(1.5, -0.1, "> 0", ha="center", va="center")
    legend_ax.text(
        -0.1,
        0.5,
        "< median",
        ha="center",
        va="center",
        rotation="vertical",
    )
    legend_ax.text(
        -0.1,
        1.5,
        "> median",
        ha="center",
        va="center",
        rotation="vertical",
    )

    # Add axis direction labels
    legend_ax.text(1, -0.3, "TOXCONC", ha="center", va="center")
    legend_ax.text(
        -0.3,
        1,
        "Population Share",
        ha="center",
        va="center",
        rotation="vertical",
    )

    # Hide ticks and axis
    legend_ax.set_xticks([])
    legend_ax.set_yticks([])
    legend_ax.axis("off")

In [ ]:
study_period = "2018_2022"

# load and merge the data
huc12 = gpd.read_file(root_dir + "results/aoi_huc12_boundaries.gpkg")
huc12_rsei = pd.read_csv(
    root_dir + "results/huc12_rsei_toxconc_weighted.csv", dtype={"huc12": str}
)
# combine tabular and spatial data using huc12 ids
huc12 = huc12.merge(huc12_rsei, on="huc12")
huc12_demographics = pd.read_csv(
    root_dir + "results/huc12_demographics.csv", dtype={"huc12": str}
)
# combine tabular and spatial data using huc12 ids
huc12 = huc12.merge(huc12_demographics, on="huc12")
huc12 = huc12.fillna(0)

us_states_gdf = gpd.read_file(
     f"zip://{root_dir}/data_input/state_boundaries/cb_2018_us_state_20m.zip/cb_2018_us_state_20m.shp"
)

In [ ]:
# Corrected color mapping
color_mapping = {
    "0-0": "#cbbacb",  # low-low
    "0-1": "#5ac8c8",  # low-high
    "1-0": "#be64ac",  # high-low
    "1-1": "#3b4994",  # high-high
}

fig, axs = plt.subplots(
    1,
    4,
    figsize=(7, 5.1),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1, 1, 1, 0.9]},
)

map_axes = axs[:3]
legend_ax = axs[3]

variables = ["share_hispanic", "share_black", "share_below_poverty"]
titles = ["Hispanic or Latino", "Black or African American", "Below Poverty"]

huc12["x_group"] = huc12[[f"{study_period}_TOXCONC"]].apply(
    mapclassify.UserDefined.make(rolling=True, bins=[0])
)

for index, var in enumerate(variables):
    us_states_gdf.to_crs(huc12.crs).plot(
        ax=map_axes[index], edgecolor="#adb5bd", facecolor="#f8f9fa", lw=1.0
    )
    huc12["y_group"] = huc12[[f"{study_period}_{var}"]].apply(
        mapclassify.Quantiles.make(rolling=True, k=2)
    )
    huc12["xy_group"] = (
        huc12["x_group"].astype(str) + "-" + huc12["y_group"].astype(str)
    )
    huc12.plot(
        categorical=True,
        linewidth=0.1,
        ax=map_axes[index],
        color=huc12["xy_group"].map(color_mapping),
    )
    wrapped_title = "\n".join(textwrap.wrap(titles[index], width=26))
    map_axes[index].set_title(wrapped_title)
    map_axes[index].set_xticks([])
    map_axes[index].set_yticks([])

# Set axis limits to match the bounds exactly
minx, miny, maxx, maxy = huc12.total_bounds
for ax in map_axes:
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

insert_bivariate_legend(legend_ax, color_mapping)

plt.show()

fig.savefig(f"{root_dir}/figures/Figure S4.pdf")